# Python Testing — Expert Interview Guide

Covers: `unittest`, `pytest`, mocking, fixtures, parametrize, TDD, async testing.

> **Key insight:** Hard-to-test code is hard-to-understand code. Testing is a design tool.

## 1. `unittest` — Built-in Test Framework

Lifecycle: `setUpClass` -> `setUp` -> `test_*` -> `tearDown` -> `tearDownClass`

In [ ]:
import unittest

def divide(a, b):
    if b == 0: raise ZeroDivisionError('Cannot divide by zero')
    return a / b

def is_palindrome(s):
    s = s.lower().replace(' ', '')
    return s == s[::-1]

class TestDivide(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        print('\n[setUpClass] runs ONCE')
        cls.shared = [1, 2, 3]

    def setUp(self):
        self.default = 10.0  # runs before EACH test

    def test_basic(self):
        self.assertEqual(divide(10, 2), 5.0)
        self.assertAlmostEqual(divide(1, 3), 0.333, places=3)

    def test_zero_raises(self):
        with self.assertRaises(ZeroDivisionError): divide(5, 0)
        with self.assertRaisesRegex(ZeroDivisionError, 'Cannot divide'):
            divide(5, 0)

    def test_palindrome(self):
        self.assertTrue(is_palindrome('racecar'))
        self.assertTrue(is_palindrome('A man a plan a canal Panama'))
        self.assertFalse(is_palindrome('hello'))

    def test_assertions_reference(self):
        self.assertEqual(1+1, 2); self.assertNotEqual(1, 2)
        self.assertIsNone(None); self.assertIsNotNone(0)
        self.assertIn('a', ['a','b']); self.assertIsInstance([], list)

suite = unittest.TestLoader().loadTestsFromTestCase(TestDivide)
unittest.TextTestRunner(verbosity=2).run(suite)

> **Interview Insight:** `setUpClass` is a `@classmethod` (receives `cls`). Use it for expensive shared setup like DB connections. `setUp` runs per-test — use it for fresh test isolation.

## 2. `pytest` — Modern Testing

No class required. Auto-discovers `test_*.py`. Rich assertion introspection shows exactly what differed.

In [ ]:
import pytest

def add(x, y): return x + y

def test_add():
    assert add(2, 3) == 5
    assert add(-1, 1) == 0

def test_exceptions():
    with pytest.raises(ZeroDivisionError): 1/0
    with pytest.raises(ZeroDivisionError, match='division by zero'): 1/0

def test_float_approx():
    assert 0.1 + 0.2 == pytest.approx(0.3)
    assert 1.0/3 == pytest.approx(0.333, abs=1e-3)

# Run manually
for name, fn in [('test_add', test_add),
                  ('test_exceptions', test_exceptions),
                  ('test_float_approx', test_float_approx)]:
    try: fn(); print(f'PASSED: {name}')
    except Exception as e: print(f'FAILED: {name} - {e}')

print()
print('Run with: pytest -v -s')

> **Interview Insight:** pytest's assertion rewriting shows `assert [1,2,3] == [1,2,4]` with a diff — which element differs. `unittest` needs `assertEqual` for good messages. pytest's `--tb=short` is great for CI.

## 3. Fixtures — Reusable Setup/Teardown

Scopes: `function` (default) | `class` | `module` | `session`

In [ ]:
import pytest

@pytest.fixture
def sample_user():
    return {'id': 1, 'name': 'Alice', 'email': 'alice@example.com'}

@pytest.fixture
def temp_store():
    store = {'items': []}
    print('\n[FIXTURE] Store created')
    yield store
    store.clear()  # teardown
    print('[FIXTURE] Store cleared')

@pytest.fixture
def make_user():
    def _make(name='Alice', role='user'):
        return {'name': name, 'role': role, 'active': True}
    return _make

# Tests using fixtures (pytest injects by parameter name)
def test_user_email(sample_user):
    assert '@' in sample_user['email']

def test_store_insert(temp_store):
    temp_store['items'].append('apple')
    assert len(temp_store['items']) == 1

def test_make_admin(make_user):
    admin = make_user('Carol', role='admin')
    assert admin['role'] == 'admin'

# Built-in fixtures
def test_tmp_path(tmp_path):
    f = tmp_path / 'test.txt'
    f.write_text('hello')
    assert f.read_text() == 'hello'

def test_monkeypatch(monkeypatch):
    import os
    monkeypatch.setenv('MY_VAR', 'test')
    assert os.environ['MY_VAR'] == 'test'

# Simulate fixture injection manually
user = {'id':1,'name':'Alice','email':'a@b.com'}
test_user_email(user)
print('test_user_email: PASSED')

> **Interview Insight:** Fixture names are matched by **parameter name** — pytest auto-injects them. No import needed. `scope='session'` fixtures are perfect for DB connections shared across the entire test run.

## 4. `@pytest.mark.parametrize` — Multiple Inputs

In [ ]:
import pytest

def is_prime(n):
    if n < 2: return False
    for i in range(2, int(n**0.5)+1):
        if n % i == 0: return False
    return True

@pytest.mark.parametrize('n,expected', [
    (2, True), (3, True), (4, False), (17, True),
    (1, False), (0, False), (-5, False),
])
def test_is_prime(n, expected):
    assert is_prime(n) == expected

@pytest.mark.parametrize('a,b,result', [
    (1, 2, 3), (-1, 1, 0), (0, 0, 0),
], ids=['positive', 'zero_sum', 'zeros'])
def test_add_param(a, b, result):
    assert a + b == result

# Manual run to demonstrate
cases = [(2, True), (4, False), (17, True), (1, False)]
for n, expected in cases:
    status = 'PASSED' if is_prime(n) == expected else 'FAILED'
    print(f'{status}: is_prime({n}) == {expected}')

> **Interview Insight:** Use `ids=` for meaningful test names: `test_add[positive]` instead of `test_add[1-2-3]`. This is critical for readable CI logs. Two stacked `@parametrize` decorators give cartesian product.

## 5. Mocking — `unittest.mock`

In [ ]:
from unittest.mock import Mock, MagicMock, patch, create_autospec
import unittest

# Basic Mock
m = Mock()
m.method(1, 2, key='val')
print(f'called:  {m.method.called}')
print(f'count:   {m.method.call_count}')
print(f'args:    {m.method.call_args}')
m.method.assert_called_once_with(1, 2, key='val')

# return_value
m2 = Mock()
m2.fetch.return_value = {'data': [1, 2, 3]}
print(f'return:  {m2.fetch()}')

# side_effect: sequence then exception
m3 = Mock()
m3.get.side_effect = [10, 20, ValueError('done')]
print(m3.get()); print(m3.get())
try: m3.get()
except ValueError as e: print(f'exception: {e}')

# MagicMock for dunders
mm = MagicMock()
mm.__len__.return_value = 5
mm.__getitem__.return_value = 'item'
print(f'len: {len(mm)}, mm[0]: {mm[0]}')

# patch decorator
class DB:
    def query(self, sql): pass

class Repo:
    def __init__(self, db): self.db = db
    def find(self, uid):
        rows = self.db.query(f'SELECT * WHERE id={uid}')
        return rows[0] if rows else None

class TestRepo(unittest.TestCase):
    def test_find_found(self):
        mock_db = create_autospec(DB, instance=True)
        mock_db.query.return_value = [{'id':1,'name':'Alice'}]
        repo = Repo(mock_db)
        result = repo.find(1)
        self.assertEqual(result['name'], 'Alice')
        mock_db.query.assert_called_once()

    def test_find_missing(self):
        mock_db = create_autospec(DB, instance=True)
        mock_db.query.return_value = []
        self.assertIsNone(Repo(mock_db).find(999))

suite = unittest.TestLoader().loadTestsFromTestCase(TestRepo)
unittest.TextTestRunner(verbosity=2).run(suite)

> **Interview Insight:** Patch **where it's used**, not where it's defined. `create_autospec` catches wrong arguments — plain `Mock()` silently accepts any call. Always prefer `create_autospec` for mocking real classes.

## 6. TDD — Red-Green-Refactor

1. **Red**: write failing test

2. **Green**: minimum code to pass

3. **Refactor**: clean up keeping tests green

In [ ]:
import unittest

# STEP 2 (GREEN): Implementation driven by tests
class Stack:
    def __init__(self): self._items = []
    def push(self, item): self._items.append(item)
    def pop(self):
        if self.is_empty(): raise IndexError('pop from empty')
        return self._items.pop()
    def peek(self):
        if self.is_empty(): raise IndexError('peek at empty')
        return self._items[-1]
    def is_empty(self): return len(self._items) == 0
    def __len__(self): return len(self._items)

# STEP 1 (RED): Tests written first
class TestStack(unittest.TestCase):
    def setUp(self): self.stack = Stack()

    def test_new_is_empty(self):
        self.assertTrue(self.stack.is_empty())
        self.assertEqual(len(self.stack), 0)

    def test_push_nonempty(self):
        self.stack.push(1)
        self.assertFalse(self.stack.is_empty())

    def test_pop_lifo(self):
        self.stack.push(1); self.stack.push(2)
        self.assertEqual(self.stack.pop(), 2)
        self.assertEqual(self.stack.pop(), 1)

    def test_pop_empty_raises(self):
        with self.assertRaises(IndexError): self.stack.pop()

    def test_peek_no_remove(self):
        self.stack.push(42)
        self.assertEqual(self.stack.peek(), 42)
        self.assertEqual(len(self.stack), 1)  # still there

suite = unittest.TestLoader().loadTestsFromTestCase(TestStack)
unittest.TextTestRunner(verbosity=2).run(suite)

> **Interview Insight:** TDD's real benefit is **design feedback** — hard-to-test code has too many dependencies or responsibilities. If writing a test is painful, the code needs refactoring. The tests are a symptom; the design is the cause.